# Insights Report
**WHERE · WHEN · WHY — DELAYS IN THE ZURICH TRAM NETWORK**

---


## Inhalt

- [Kernthese](#kernthese)
- [Zentrale Fragen — Befunde](#zentrale-fragen-befunde)
- [Lever Comparison: Schedule Structure vs. External Factors](#lever-comparison-schedule-structure-vs-external-factors)
    - [Befund](#befund)
- [Netzstruktur](#netzstruktur)
  - [Wiederkehrende Einschnitte](#wiederkehrende-einschnitte)
    - [Befund](#befund)
  - [On-Time-Performance](#on-time-performance)
    - [Befund](#befund)
  - [Accumulating Delays](#accumulating-delays)
    - [Befund](#befund)
    - [Dwell Times per Line and Delays](#dwell-times-per-line-and-delays)
  - [Strukturelle Stop-Typen: Akkumulations-Fallen & Recovery-Anker](#strukturelle-stop-typen-akkumulations-fallen-recovery-anker)
- [Geografie](#geografie)
  - [Haltestellen Delay-Hotspots](#haltestellen-delay-hotspots)
    - [Befund](#befund)
  - [Das Muster entlang der Strecke: L11 vs. L6](#das-muster-entlang-der-strecke-l11-vs-l6)
    - [Befund](#befund)
  - [Mechanismus: Kein Puffer — keine Erholung](#mechanismus-kein-puffer-keine-erholung)
    - [Befund](#befund)
  - [Evidence Chain: Cascade Effect Network-Wide](#evidence-chain-cascade-effect-network-wide)
    - [Befund](#befund)
  - [Stadtkreise als Problemzonen](#stadtkreise-als-problemzonen)
    - [Befund](#befund)
- [Temporality](#temporality)
  - [Temporal Patterns](#temporal-patterns)
    - [Befund](#befund)
- [Meteorologie](#meteorologie)
  - [Wetter Ereignisse](#wetter-ereignisse)
    - [Befund](#befund)
  - [Jahreszeit](#jahreszeit)
    - [Befund](#befund)
- [Large-Scale Events](#large-scale-events)
  - [Feiertage & Events](#feiertage-events)
    - [Befund](#befund)
  - [Netzausbau 2023](#netzausbau-2023)
    - [Befund](#befund)
- [Empfehlungen](#empfehlungen)


**Fokus** Wo · Wann · Warum — Verspätungen im Zürcher Tramnetz  
**Ziel** Strukturelle Muster sichtbar machen · Basis für operative Empfehlungen

**Daten-Quelle**  
* IST-Daten von opentransportdata.swiss 
* GTFS- und Meteo-Datan von data.stadt-zuerich.ch

**Daten-Umfang**  
* Exploration: ~94,4 Mio. Zeilen · 16 Tramlinien · 26 Spalten
* Analyse: ~85,4 Mio. Zeilen · 16 Tramlinien · 42 Spalten

**Herausforderungen**
* Datenmenge (3 Jahre)
* Datenqualität (Betriebsbedingt)

**Überblick** 
* Erkenntnisse 
    * Netzstruktur
    * Geografie
    * Temporalität
    * Meteorologie
    * Ereignisse
    * Infrasturkur
* Empfehlungen

---

---

## Kernthese

> **"Die Verspätungen im Zürcher Tramnetz sind vorhersagbar, sie sind systematisch —  
> weil sie im Fahrplan-Design verankert sind, nicht im Betrieb."**

Drei strukturelle Befunde, die zusammen mehr sagen als jeder einzeln:

* **Fahrplan als Treiber:** 71.5% aller Halte akkumulieren Delay · 71.3% haben keine Pufferzeit (dwell_time = 0 s) — das Problem ist eingebaut, nicht operativ
* **Vorhersagbare Kaskade:** Pearson r ≥ 0.85 netzweit — Delay überträgt sich fast vollständig von Halt zu Halt · Ein Modell (LightGBM, MAE 45.7 s) kann das reproduzieren
* **Konsequenz:** Was vorhersagbar ist, ist steuerbar · Die Lösung liegt im Fahrplan-Design: Puffer an den richtigen Stellen — nicht im Netzausbau

---

## Zentrale Fragen — Befunde

| Frage | Befund | Abschnitt |
|:---|:---|:---|
| Wo entstehen Verspätungen? | Periphere Endhalte (K11: 68s, K8: 64s) — Zentrum besser als Randlagen | [Geografie](#geografie) |
| Zu welchen Zeiten? | Abend-Peak 21h · Donnerstag schlechtester Tag · Ferien/Feiertage messbar besser | [Temporalität](#temporalität) |
| Welche Einflussfaktoren? | Fahrplan-Design > Schnee > Events > Autoverkehr — strukturelle Ursache dominiert | [Hebel-Vergleich](#hebel-vergleich-fahrplan-struktur-vs-externe-faktoren) |
| Vorhersagbar? | Ja — LightGBM MAE 18.56s · −63% vs. Baseline | `06_prediction_4` |
| Netzausbau 2023: Wirkung? | Kein messbarer Effekt — VBZ hat den größten Netzumbau ihrer Geschichte operativ sauber abgewickelt | [Netzstruktur](#netzstruktur) |
| Hotspots und Kaskadeneffekt? | Pearson r ≥ 0.85 netzweit — alle 16 Linien im roten Bereich | [Geografie](#geografie) |
| Was kann ein Betreiber tun? | Puffer einbauen · Fahrplan-Redesign L11 · Fachmessen-Disposition | [Empfehlungen](#empfehlungen) |


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # Plotly: CDN-Link im Output → HTML-Export funktioniert

from zh_tram_flow.notebook import *
import zh_tram_flow.analytics as an
from zh_tram_flow.visualization.insights import (
    plot_monthly_delay_by_line,
    plot_otp_by_line,
    plot_otp_delta_distribution,
    plot_dwell_analysis,
    plot_dwell_throughput,
    plot_dwell_vs_delay,
    plot_district_maps,
    plot_district_delay_map,
    plot_district_otp_map,
    plot_infra_maps,
    plot_delay_delta_timeline,
    plot_arrival_vs_departure_timeline,
    plot_lever_comparison,
    plot_dwell_scatter,
    plot_dwell_line_scatter,
)

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("04_insights")

## Lever Comparison: Schedule Structure vs. External Factors

Wie schwer wiegt die strukturelle Fahrplan-Schwäche im Vergleich zu situativen Einflussfaktoren?

* **Struktureller Hebel** — `mean(delay_delta) × avg_stops_per_trip`: kumulierter Verspätungsaufbau über eine Ø Fahrt — tritt **jeden Tag**, auf jeder Linie, unabhängig vom Kontext auf
* **Externe Faktoren** — gemessener Δ Arrival Delay pro Halt vs. Normalbedingung: situativ, nur wenn Bedingung eintrifft (Schnee, Event, Tageszeit)

#### Befund

Der Fahrplan selbst ist der grösste Hebel — grösser als Schnee, grösser als Grossereignisse, grösser als der Abend-Peak. Und anders als Wetter oder Events ist er steuerbar.

In [ ]:
plot_lever_comparison(lf_clean)

## Netzstruktur

* Stabile Verspätungswerte trotz großer Einschnitte
* On-Time-Performance (OTP) unter Zielvorgaben
* Akkumulierte Verspätungen durch zu wenig Puffer

### Wiederkehrende Einschnitte



**Fahrplanwechsel Dez 2023**   
Zu dem Zeitpunkt grösster Netzumbau der VBZ-Geschichte: 10 von 17 Linien umgebaut, L9/L11/L13 mit neuen Streckenführungen und Haltestellen.    

**Groß-Baustelle Ende im Jun 2024**     
L12 Streckensperrung mit Ersatzverkehr (Jan 2023–Jun 2024). Ausfallrate 20× über Netzschnitt während der Bauphase.    

**Fahrplanwechsel Dez 2024**    
Stabiler Übergang, geringer Eingriff (L5: +5 Halte).  

**Fahrplanwechsel Dez 2025**    
Grösster Fahrplanwechsel der VBZ-Geschichte: 7 von 18 Linien signifikant umgebaut. Neu: „Tramnetz Süd" mit L50/L51 als provisorische Baustellen-Linien für die Bahnhofquai/HB-Sanierung (bis Dez 2026).


#### Befund

**Kein erkennbarer Effekt auf das Verspätungsverhalten**     
* Alle drei Markierungen hinterlassen keinen erkennbaren Knick im Verspätungsverlauf.    
* Veränderte Linien (L11 +5.3s nach Umbau) bewegen sich identisch zu stabilen Referenzlinien (L15 +5.2s).     
* Die VBZ hat erhebliche Netzeingriffe operativ sauber abgewickelt — das Verspätungsniveau folgt saisonalen Mustern, nicht betrieblichen Ereignissen.

In [ ]:
plot_monthly_delay_by_line(lf_clean)

### On-Time-Performance



OTP-Definitio der VBZ:    
**"Anteil aller Ankünfte innerhalb von ±120 Sekunden (±2 Min.) des Fahrplans."**

**OTP Werte netzweit** 
* Über alle Tram-Linien hinweg wird eine OTP von 87% erreicht.
* Bester Wert liegt bei 93,4% der Linie 6
* Schlechtester Wert liegt bei 81,6% der Linie 11

#### Befund
* Das Netz liegt mit 87% rund 8 Prozentpunkte unter dem VBZ-Ziel von 95%.
* Die Spanne reicht von 81.6% (L11) bis 93.4% (L6) — alle Linien verfehlen das Ziel.

In [ ]:
plot_otp_by_line(lf_clean)

### Accumulating Delays



**71.5% aller Halte** — Verspätung wächst Stop zu Stop

* 27.2% bauen Verspätung ab · 1.3% neutral
* 71.3% aller Halte haben keine Verweilzeit (dwell_time = 0s) und dadurch kein Puffer zum Nachholen

#### Befund

* Verspätungen entstehen nicht punktuell — sie wachsen systematisch über den gesamten Trip
* Fehlende Pufferzeit ist die Ursache: ohne dwell_time kann das Tram verlorene Zeit nicht aufholen
* Das ist ein Fahrplan-Problem, kein Kapazitätsproblem — die Lösung liegt im Design, nicht im Ausbau



In [ ]:
plot_otp_delta_distribution(lf_clean)

In [ ]:
plot_dwell_throughput(lf_clean)

#### Dwell Times per Line and Delays

* **Ungleich verteilt** — Verweilzeiten sind sehr unterschiedlich auf die Linien verteilt
* **L3** liegt im Verspätungs-Durchschnitt — offensichtlich getragen durch überdurchschnittliche Verweilzeiten
* **L11** könnte durch mehr Verweilzeiten in den Durchschnitt gebracht werden — strukturell unterversorgt
* **L6** hat so viel Verweilzeit, dass die Verspätungen unterdurchschnittlich gering sind — positives Gegenbeispiel

In [ ]:
plot_dwell_vs_delay(lf_clean)

### Strukturelle Stop-Typen: Akkumulations-Fallen & Recovery-Anker

Nicht alle Stops sind gleich — die Daten zeigen zwei klar trennbare Typen:

* **Akkumulations-Fallen** — dw0 ≥ 90%, hoher delay_delta: kein Puffer, jede Sekunde Verspätung die ankommt wird weitergegeben und verschlimmert. Bsp: Enzenbühl (94.6%, +9.5s/Halt), Mattenhof (95.5%, +11.9s/Halt), Messe/Hallenstadion (98.4%, +19.7s/Halt)
* **Recovery-Anker** — dw0 ≤ 50%, negativer delay_delta: echter geplanter Puffer, Trams bauen hier aktiv Verspätung ab. Bsp: Seebacherplatz (42.7%, −17.2s/Halt), Oerlikon (44.3%, −7.8s/Halt)

**Kernbefund:** Recovery-Anker kommen trotz Abbau mit 80–100s an — weil der Vorlauf so gross ist, dass selbst 34s Puffer nicht ausreicht. Das Netz versucht sich zu reparieren. Es scheitert am systematischen Vorlauf.

In [ ]:
plot_dwell_scatter(lf_clean)

In [ ]:
plot_dwell_line_scatter(lf_clean)

## Geografie

### Haltestellen Delay-Hotspots


**Periphere Aussenkorridore** — nicht zentrale Knotenpunkte

* Friedhof Enzenbühl: 93.8 s · Balgrist: 85.2 s · Leutschenbach: 82.7 s
* Central: 48.3 s (15 Linien) · Paradeplatz: 48.2 s (14 Linien) — unter dem Netzschnitt

**Strukturell schwächste Zonen**

* K11: 68.3 s · OTP 83% · K12: 66.3 s · K8: 63.7 s

#### Befund

* Hotspots liegen am Stadtrand — viel befahrene Innenstadtknoten performen besser als periphere Endpunkte
* Paradeplatz und Central (je 14–15 Linien) bestätigen: Frequenz allein ist kein Verspätungstreiber

In [ ]:
an.plot_stop_delay_map(lf_clean)

### Das Muster entlang der Strecke: L11 vs. L6

Die Hotspot-Karte zeigt **wo** die Verspätung am höchsten ist — aber nicht **warum sie dort sitzt**.

Ein direkter Vergleich der schlechtesten und besten Linie macht das Muster sichtbar:

* **L11** (Rehalp–Auzelg · 30.2 km · OTP 81.6% · ⌀ 71 s) — Bubbles wachsen konsequent von der Mitte zur Peripherie: Verspätung akkumuliert entlang der Strecke und gipfelt an den Endhalten
* **L6** (HB–Zoo · OTP 93.4% · ⌀ 38 s) — Bubbles bleiben über die gesamte Strecke klein: eingebauter Puffer bricht die Kaskade

#### Befund

Hotspots sind **nicht** das Ergebnis schlechter Infrastruktur an diesen Punkten — sie sind das **Endprodukt einer Kettenreaktion**, die an der ersten Station beginnt. Der Unterschied zwischen L11 und L6 ist keine Frage des Glücks, sondern des Fahrplan-Designs.

In [ ]:
an.plot_line_delay_profile_map(lf_clean, lines=["11", "6"], cfg=cfg)

### Mechanismus: Kein Puffer — keine Erholung

Warum kann L11 die angesammelte Verspätung nicht abbauen?

Die Dwell-Map zeigt die kritische Variable:
* **Farbe** — Ø Arrival Delay (Grün = pünktlich · Rot = verspätet)
* **Bubble-Grösse** — Anteil der Abfahrten **ohne** Pufferzeit (dwell_time = 0 s)

**Smoking Gun:** Die rötesten Bubbles sind gleichzeitig die grössten — die Haltestellen mit den schlimmsten Verspätungen haben den wenigsten Puffer. Kein Spielraum zum Aufholen. Die Verspätung, die ein Tram mitbringt, gibt es komplett weiter — und legt noch etwas drauf.

#### Befund

Das ist kein operatives Problem, das ein Fahrer lösen kann. Es ist ein **Fahrplan-Problem**: fehlende dwell_time ist eingebaut. Wer das beheben will, muss das Fahrplan-Design ändern — nicht den Betrieb.

In [ ]:
an.plot_stop_dwell_map(lf_clean, line_name="11")

### Evidence Chain: Cascade Effect Network-Wide

Ist L11 ein Einzelfall — oder ist das ein systemisches Muster?

Der **Kaskadenkoeffizient** misst, wie stark sich die Verspätung eines Halts auf den nächsten überträgt (Pearson r):
* r → 1.0 = Delay kaskadiert vollständig — das Tram trägt jede Sekunde Verspätung einfach weiter
* r → 0.0 = Das Netz erholt sich — Verspätungen sind zufällig, kein Durchreichen

#### Befund

Alle 16 Linien liegen im roten Bereich (r ≥ 0.85). Das ist keine statistische Streuung — es ist der **Fingerabdruck des Fahrplan-Designs**. Ein Netz, das sich selbst reparieren könnte, würde hier Werte unter 0.70 zeigen. Kein einziges Tram schafft das.

Die vier Schritte der Beweiskette sind damit geschlossen:
1. **Anomalie** — Hotspots liegen systematisch an peripheren Endhalten
2. **Gradient** — Delay wächst entlang der Strecke (L11 vs. L6 als Kontrast)
3. **Mechanismus** — Kein Puffer (dwell_time = 0 s) → keine Erholung möglich
4. **Beweis** — Pearson r ≥ 0.85 netzweit: systematische Kaskade, kein Zufall

In [ ]:
an.plot_cascade_effect(lf_clean, ylim=(0.8, 0.95))

### Stadtkreise als Problemzonen



**Problemzone 1** — K11 · K12 · K8

* K11: 68.3s · OTP 83% · K12: 66.3s · OTP 85% · K8: 63.7s · OTP 85%
* Alle drei liegen 8–18s über dem Netzschnitt — strukturell schwächste Zone

**Problemzone 2** — K9 · K7 · Aussenbezirk

* K9: 59.7s · K7: 58.7s · Aussenbezirk: 58.4s — OTP jeweils 87%
* Erkennbarer Abstand zu Gruppe 1, aber deutlich über den besten Kreisen

**Beste Kreise** — K5 · K10 · K1

* K5: 49.9s · OTP 89% — bester Kreis netzweit
* K1 (Paradeplatz / Innenstadt): 51.3s — bester Innenstadtknoten mit höchster Haltestellendichte (18.4 Mio. Beobachtungen)

#### Befund

* Zwei klar abgrenzbare Problemzonen — kein gleichmässiges Gefälle über das Netz
* Zentrale Knotenpunkte (K1: 51.3s) performen besser als periphere Aussenkorridore (K11: 68.3s) — Dichte ist kein Problem, Lage ist ein Problem
* OTP-Spanne zwischen bestem (K5: 89%) und schlechtestem Kreis (K11: 83%): 6 Prozentpunkte

In [ ]:
an.plot_district_combined(lf_clean, ylim_otp=(80, 91))

## Temporality


### Temporal Patterns

**Abend-Peak** — 21h · 67.9s

* Höchste Verspätung des Tages — Abreisewellen nach Events und Abendveranstaltungen
* Kein Morgenrush: 7h = 48.9s — unter dem Netzschnitt

**Wochentag**

* Schlechtester Tag: Donnerstag · 60.4s · P95 = 194s
* Bester Tag: Sonntag · 48.4s · Montag · 52.3s — Homeoffice-Effekt sichtbar

> **P95** — 95. Perzentil der Ankunftsverspätung: 95% aller Halte liegen *unter* diesem Wert, nur die schlechtesten 5% darüber. Zeigt Extremverspätungen, nicht den Alltag.

#### Befund

* Abend dominiert — kein klassischer Morgenrush im Zürcher Tramnetz
* Wochenende und Montag profitieren von reduziertem MIV — direkter Zusammenhang mit Kfz-Verkehrsdichte
* P95 am Donnerstag (194s): an schlechten Tagen warten Fahrgäste über 3 Minuten auf das Tram

In [ ]:
an.plot_hour_of_day(lf_clean, cfg, ylim=(0,80), ylim_volume=(0, 6.5))

an.plot_day_of_week(lf_clean, cfg, ylim=(0, 70), ylim_otp=(85, 95))


## Meteorologie

* Wetter ereignisse sind deutlich nachweisbar
* Regen, Starkregen und Schnee wirken sich aufsteigend aus
* Regen/Starkregen ist eher ein Problematik des Zentrums, K5
* Schnee hat ehr Einfluss auf die äusseren Stadtkreise, K10

### Wetter Ereignisse



**Stärkster Einzelfaktor** — Schnee · +54s · OTP −10.9pp

* Regen: +14s · Starkregen: +22s — deutlich schwächer als Schnee

**Geografische Trennung**

* Schnee → Höhenlagen: K10 · K4 · K12
* Starkregen → Flusstäler: K5 (Escher Wyss / Toni-Areal)

**Linien-Paradox**

* L17: Schnee +7.7s · Regen +41.2s — reagiert umgekehrt zur Netztendenz
* L9: Schnee +75.9s · Regen +10.0s — extremste Schnee-Betroffenheit im Netz

#### Befund

* Schnee ist der einzige Wetterfaktor mit zweistelligem OTP-Einbruch (−10.9pp)
* Wetter trifft nicht alle Linien gleich — geografische Lage der Strecke entscheidet über Exposition
* Regen allein ist kein kritischer Faktor — erst Starkregen in Flusstälern zeigt nennenswerten Effekt

In [ ]:
an.plot_weather_overview(lf_clean)

In [ ]:
an.plot_weather_stop_map_combined(lf_clean)

### Jahreszeit



**Beste Jahreszeit** — Winter · 51.7s · OTP 88.9%

* Herbst: 61.2s — schlechteste Jahreszeit · Sommer: 56.4s · Frühling: 55.6s
* Weniger MIV im Winter → weniger Kreuzungskonflikte — übertrifft den Schnee-Effekt

#### Befund

* Winter ist trotz Schnee die beste Jahreszeit — reduzierter Kfz-Verkehr überwiegt den Schnee-Malus
* Saisonspanne von 9.5s (Winter → Herbst) ist strukturell stabil über alle drei Messjahre
* Herbst kombiniert volles Verkehrsaufkommen mit wechselhaftem Wetter — belastendste Jahreszeit

In [ ]:
an.plot_month_seasonality(lf_delay, cfg, panel="left")

## Large-Scale Events


* Besonders die Großereignisse fallen ins Gewicht, ab 20k
* Fachmessen und Kongresse sind kritischer als Konzerte
* Events wirken deutlich eher am abends 

### Feiertage & Events



**Feiertage** — Bester Tag-Typ · 46.3s · OTP 90.6%

* −9.9s gegenüber einem normalen Werktag — reduzierter MIV als Haupttreiber
* Grosse Events: +10.5s — fast ausschliesslich abends 18–22h
* Tagsüber: Event-Tage ≈ Normaltage — kein messbarer Effekt vor 18h

In [ ]:
an.plot_events_overview(lf_clean)

**Schlechteste Kategorie** — Fachmessen · 66.0s · OTP 84%

* Schlechtester Tag: Berufsmesse Zürich 21.11.2024 · 192.5s · OTP 54.5%
* Taylor Swift: 75.4s — Fachmessen schlagen Popkonzerte deutlich

#### Befund

* Feiertage entlasten das Netz stärker als jeder andere Faktor — 46.3s unterschreitet sogar den besten Kreisschnitt
* Fachmessen sind kritischer als Konzerte — andere Zielgruppe, andere Anreisezeit, weniger ÖPNV-Affinität
* Events wirken fast ausschliesslich abends — tageszeitliche Steuerung ist die wirksamste Gegenmassnahme

In [ ]:
plot_arrival_vs_departure_timeline(lf_clean)

### Netzausbau 2023

**Ausgebaut** — K3 · K8

* Sihlcity (K3): 56.0 s · Rehalp (K8): 63.8 s

**Nicht ausgebaut** — K11 · K12

* K11: 68.3 s · OTP 83% · K12: 66.3 s

#### Befund

Der Netzausbau Dez 2023 hat die Verspätungsstruktur **nicht verändert** — weder bei den ausgebauten Kreisen (K3/K8) noch bei den übrigen. Im monatlichen Timeline-Plot ist kein Knick sichtbar. Ausgebaute Linien wie L11 (+5.3 s nach Umbau) entwickeln sich identisch zu stabilen Referenzlinien (L15 +5.2 s).

Das ist kein Versagen — es ist ein Befund: **Infrastrukturausbau löst kein Fahrplan-Design-Problem.**

Wenn der grösste Netzumbau der VBZ-Geschichte keine messbare Wirkung auf das Verspätungsniveau hinterlässt, liegt die Ursache nicht in der Infrastruktur. Sie liegt in der Fahrplanstruktur: fehlende Pufferzeiten, die sich bei jeder Fahrt von Halt zu Halt aufschaukeln.

In [ ]:
plot_infra_maps(lf_delay, lf_clean)

## Empfehlungen

**Strukturelle Muster** — stabil über 3 Jahre

| Priorität | Empfehlung | Basis |
|:---|:---|:---|
| 🔴 Hoch | Puffer einbauen — 10 s dwell_time an Aussenkorridor-Halten (Balgrist, Enzenbühl, Leutschenbach) | 71.5% akkumulieren · 71.3% dwell_time = 0 s · Kaskadeneffekt r ≥ 0.85 |
| 🔴 Hoch | Fahrplan-Redesign L11 — explizite Pufferzeiten an End- und Problemhalten | OTP 81.6% · ⌀ 71 s · Dwell-Map: röteste Stops haben 0 s Puffer |
| 🟡 Mittel | Fachmessen-Disposition — L11 Verstärkerkurse an Berufsmesse-Tagen | Schlechtester Tag: 192.5 s · OTP 54.5% — vorhersagbar, planbar |
| 🟡 Mittel | Schnee-Protokoll Höhenlagen K10/K4 — präventive Massnahmen bei Schneeprognose | L9/L12 bis +76 s bei Schnee · Selnau Extremfall +190.9 s |